In [14]:
# ============================================================================
# CELL 1: IMPORTS & SETUP
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


In [15]:
# ============================================================================
# CELL 2: LOAD COMPETITION DATA (Local Files)
# ============================================================================

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print("Competition Train shape :", train.shape)
print("Competition Test shape  :", test.shape)
print("Sample submission shape :", sample_sub.shape)

print("\nTarget Distribution:")
print(train['Churn'].value_counts())
print(train['Churn'].value_counts(normalize=True))

Competition Train shape : (594194, 21)
Competition Test shape  : (254655, 20)
Sample submission shape : (254655, 2)

Target Distribution:
Churn
No     460377
Yes    133817
Name: count, dtype: int64
Churn
No     0.774792
Yes    0.225208
Name: proportion, dtype: float64


In [16]:
# ============================================================================
# CELL 3: LOAD ORIGINAL TELCO DATASET
# ============================================================================
# Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
# Place file as: WA_Fn-UseC_-Telco-Customer-Churn.csv in the same folder

try:
    original = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
    print(f"Original dataset loaded: {original.shape}")
except FileNotFoundError:
    print("⚠️ Original dataset not found. Using only competition data.")
    print("For best score, download WA_Fn-UseC_-Telco-Customer-Churn.csv")
    original = None

⚠️ Original dataset not found. Using only competition data.
For best score, download WA_Fn-UseC_-Telco-Customer-Churn.csv


In [17]:
# ============================================================================
# CELL 4: PREPARE & MERGE ORIGINAL DATA
# ============================================================================

if original is not None:
    # Fix column name
    if 'customerID' in original.columns:
        original = original.drop(columns=['customerID'])
    
    # Fix TotalCharges (contains blank strings)
    original['TotalCharges'] = pd.to_numeric(original['TotalCharges'], errors='coerce')
    original['TotalCharges'] = original['TotalCharges'].fillna(0.0)
    
    # Add dummy id
    original['id'] = -999
    
    # Align columns
    common_cols = [col for col in train.columns if col in original.columns]
    train_combined = pd.concat([train[common_cols], original[common_cols]], 
                               axis=0, ignore_index=True)
    
    print(f"Combined training rows: {len(train_combined):,}")
else:
    train_combined = train.copy()
    print("Using only competition train data")

Using only competition train data


In [18]:
# ============================================================================
# CELL 5: FEATURE ENGINEERING
# ============================================================================

def feature_engineering(df):
    df = df.copy()
    
    # Convert target to numeric
    if 'Churn' in df.columns and df['Churn'].dtype == 'object':
        df['Churn'] = (df['Churn'] == 'Yes').astype(int)
    
    # Numeric interactions
    df['tenure_monthly_ratio'] = df['tenure'] / (df['MonthlyCharges'] + 1e-6)
    df['total_charges_per_month'] = df['TotalCharges'] / (df['tenure'] + 1)
    df['monthly_vs_total'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1e-6)
    df['tenure_squared'] = df['tenure'] ** 2
    df['charges_diff'] = df['MonthlyCharges'] - df['total_charges_per_month']
    
    # Tenure bins
    df['tenure_bin'] = pd.cut(df['tenure'], 
                              bins=[-1, 6, 12, 24, 48, 72, 999],
                              labels=[0,1,2,3,4,5]).astype(int)
    
    df['is_new'] = (df['tenure'] <= 6).astype(int)
    df['is_loyal'] = (df['tenure'] >= 48).astype(int)
    
    # Service count
    service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                    'TechSupport', 'StreamingTV', 'StreamingMovies']
    for col in service_cols:
        if col in df.columns:
            df[col + '_flag'] = (df[col] == 'Yes').astype(int)
    
    flag_cols = [c for c in df.columns if c.endswith('_flag')]
    df['total_services'] = df[flag_cols].sum(axis=1)
    
    # Bundles
    df['streaming_bundle'] = ((df['StreamingTV'] == 'Yes') & 
                              (df['StreamingMovies'] == 'Yes')).astype(int)
    df['security_bundle'] = ((df['OnlineSecurity'] == 'Yes') & 
                             (df['DeviceProtection'] == 'Yes')).astype(int)
    
    # Label encode categorical columns
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        if col != 'Churn':
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
    
    return df

# Apply feature engineering
train_fe = feature_engineering(train_combined)
test_fe = feature_engineering(test)

print(f"Train FE shape: {train_fe.shape}")
print(f"Test FE shape : {test_fe.shape}")

Train FE shape: (594194, 38)
Test FE shape : (254655, 37)


In [19]:
# ============================================================================
# CELL 6: PREPARE X, y, X_test
# ============================================================================

drop_cols = ['id', 'Churn']
feature_cols = [col for col in train_fe.columns if col not in drop_cols]

# Ensure test has same columns
X = train_fe[feature_cols].copy()
y = train_fe['Churn'].copy()
X_test = test_fe[feature_cols].copy()

print(f"Final feature count : {len(feature_cols)}")
print(f"Training samples    : {len(X):,}")

Final feature count : 36
Training samples    : 594,194


In [20]:
# ============================================================================
# CELL 7: XGBOOST 5-FOLD CV
# ============================================================================

print("🚀 Training XGBoost...")

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',        # Change to 'gpu_hist' if you have GPU
    'learning_rate': 0.02,
    'max_depth': 7,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': SEED,
}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

xgb_oof = np.zeros(len(X))
xgb_test_pred = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    dtest = xgb.DMatrix(X_test)
    
    model = xgb.train(
        xgb_params, 
        dtrain,
        num_boost_round=3000,
        evals=[(dval, 'valid')],
        early_stopping_rounds=150,
        verbose_eval=False
    )
    
    xgb_oof[val_idx] = model.predict(dval)
    xgb_test_pred += model.predict(dtest) / N_FOLDS
    
    fold_score = roc_auc_score(y_val, xgb_oof[val_idx])
    print(f"Fold {fold} AUC: {fold_score:.6f}")

xgb_score = roc_auc_score(y, xgb_oof)
print(f"\nXGBoost OOF Score: {xgb_score:.6f}")

🚀 Training XGBoost...
Fold 1 AUC: 0.916041
Fold 2 AUC: 0.917102
Fold 3 AUC: 0.916643
Fold 4 AUC: 0.917631
Fold 5 AUC: 0.914735

XGBoost OOF Score: 0.916420


In [21]:
# ============================================================================
# CELL 8: LIGHTGBM 5-FOLD CV
# ============================================================================

print("🚀 Training LightGBM...")

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.02,
    'num_leaves': 63,
    'max_depth': 8,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': SEED,
    'verbose': -1,
    'device': 'cpu'               # Change to 'gpu' if available
}

lgb_oof = np.zeros(len(X))
lgb_test_pred = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    train_set = lgb.Dataset(X_train, label=y_train)
    val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)
    
    model = lgb.train(
        lgb_params,
        train_set,
        num_boost_round=3000,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(150, verbose=False)]
    )
    
    lgb_oof[val_idx] = model.predict(X_val)
    lgb_test_pred += model.predict(X_test) / N_FOLDS
    
    fold_score = roc_auc_score(y_val, lgb_oof[val_idx])
    print(f"Fold {fold} AUC: {fold_score:.6f}")

lgb_score = roc_auc_score(y, lgb_oof)
print(f"\nLightGBM OOF Score: {lgb_score:.6f}")

🚀 Training LightGBM...
Fold 1 AUC: 0.915941
Fold 2 AUC: 0.917044
Fold 3 AUC: 0.916378
Fold 4 AUC: 0.917466
Fold 5 AUC: 0.914795

LightGBM OOF Score: 0.916314


In [22]:
# ============================================================================
# CELL 9: CATBOOST 5-FOLD CV
# ============================================================================

print("🚀 Training CatBoost...")

cat_oof = np.zeros(len(X))
cat_test_pred = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.02,
        depth=7,
        l2_leaf_reg=3,
        eval_metric='AUC',
        random_seed=SEED,
        verbose=0,
        task_type='CPU',           # Change to 'GPU' if available
        early_stopping_rounds=150
    )
    
    model.fit(X_train, y_train, eval_set=(X_val, y_val))
    
    cat_oof[val_idx] = model.predict_proba(X_val)[:, 1]
    cat_test_pred += model.predict_proba(X_test)[:, 1] / N_FOLDS
    
    fold_score = roc_auc_score(y_val, cat_oof[val_idx])
    print(f"Fold {fold} AUC: {fold_score:.6f}")

cat_score = roc_auc_score(y, cat_oof)
print(f"\nCatBoost OOF Score: {cat_score:.6f}")

🚀 Training CatBoost...
Fold 1 AUC: 0.916072
Fold 2 AUC: 0.917225
Fold 3 AUC: 0.916517
Fold 4 AUC: 0.917632
Fold 5 AUC: 0.914863

CatBoost OOF Score: 0.916453


In [23]:
# ============================================================================
# CELL 10: ENSEMBLE & SUMMARY
# ============================================================================

# Equal weight ensemble
ensemble_oof = (xgb_oof + lgb_oof + cat_oof) / 3.0
ensemble_test = (xgb_test_pred + lgb_test_pred + cat_test_pred) / 3.0

ensemble_score = roc_auc_score(y, ensemble_oof)

print("="*70)
print("                      FINAL MODEL SUMMARY")
print("="*70)
print(f"Model           |    OOF CV Score | Notes")
print("-"*70)
print(f"XGBoost         |        {xgb_score:.6f} | GPU-accelerated GBDT")
print(f"LightGBM        |        {lgb_score:.6f} | Leaf-wise tree growth")
print(f"CatBoost        |        {cat_score:.6f} | Ordered boosting")
print("-"*70)
print(f"Ensemble        |        {ensemble_score:.6f} | Equal weight average")
print("="*70)
print(f"\nBest Model      : XGBoost with {xgb_score:.6f}" if xgb_score == max(xgb_score, lgb_score, cat_score) else "")
print(f"Ensemble Strategy: Equal weight average (1/3 each)")
print(f"Training Size    : {len(X):,} rows")
print(f"Feature Count    : {len(feature_cols)}")
print(f"Cross-Validation : {N_FOLDS}-Fold Stratified")

                      FINAL MODEL SUMMARY
Model           |    OOF CV Score | Notes
----------------------------------------------------------------------
XGBoost         |        0.916420 | GPU-accelerated GBDT
LightGBM        |        0.916314 | Leaf-wise tree growth
CatBoost        |        0.916453 | Ordered boosting
----------------------------------------------------------------------
Ensemble        |        0.916665 | Equal weight average

Ensemble Strategy: Equal weight average (1/3 each)
Training Size    : 594,194 rows
Feature Count    : 36
Cross-Validation : 5-Fold Stratified


In [24]:
# ============================================================================
# CELL 11: CREATE SUBMISSION
# ============================================================================

submission = pd.DataFrame({
    'id': test['id'],
    'Churn': ensemble_test
})

submission.to_csv('submission.csv', index=False)

print("✅ Submission saved as submission.csv")
print(f"Shape: {submission.shape}")
print(f"Churn probability range: [{submission['Churn'].min():.4f}, {submission['Churn'].max():.4f}]")
display(submission.head())

✅ Submission saved as submission.csv
Shape: (254655, 2)
Churn probability range: [0.0002, 0.9918]


,id,Churn
0,594194,0.069985
1,594195,0.000590
2,594196,0.105106
3,594197,0.003927
4,594198,0.491898
